In [7]:
from utils.spark import build_spark_explore
from utils.constants import CAPITALS_TABLE_PATH

spark = build_spark_explore()
spark.sql("SHOW TABLES").show()


+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|         hrrr_bronze|      false|
|  default|hrrr_gold_capital...|      false|
|  default|hrrr_gold_capital...|      false|
|  default|         hrrr_silver|      false|
|  default|         us_capitals|      false|
+---------+--------------------+-----------+



In [3]:
spark.sql('SELECT * FROM us_capitals').show()

+--------------+-----+-------+------------------+------------------+------------------+------------------+------------------+
|          city|state|    lat|               lon|           min_lat|           max_lat|           min_lon|           max_lon|
+--------------+-----+-------+------------------+------------------+------------------+------------------+------------------+
|    Montgomery|   AL|32.3615|273.72090000000003|           31.9615|           32.7615|273.32090000000005|          274.1209|
|        Juneau|   AK|58.3019|          225.5803|57.901900000000005|           58.7019|          225.1803|          225.9803|
|       Phoenix|   AZ|33.4484|           247.926|           33.0484|           33.8484|247.52599999999998|           248.326|
|   Little Rock|   AR|34.7361|          267.6689|           34.3361|           35.1361|267.26890000000003|          268.0689|
|    Sacramento|   CA|38.5737|          238.5129|38.173700000000004|           38.9737|          238.1129|          23

In [4]:
spark.sql("DESCRIBE TABLE hrrr_silver").show(30, truncate=False)

+-----------------------+---------+--------------------------------------------+
|col_name               |data_type|comment                                     |
+-----------------------+---------+--------------------------------------------+
|date                   |date     |Forecast Date                               |
|valid_time             |timestamp|Forecast Valid Time                         |
|latitude               |float    |Latitude (Degrees)                          |
|longitude              |float    |Longitude (Degrees, 0 to 360)               |
|sp                     |float    |Surface Pressure [Pa]                       |
|gust                   |float    |Surface Wind Gust [m/s]                     |
|vis                    |float    |Visibility [m]                              |
|prate                  |float    |Precipitation Rate [kg/m^2/s]               |
|crain                  |float    |Categorical Rain (yes=1; no=0)              |
|csnow                  |flo

In [5]:
df = spark.read.table("hrrr_silver")
# df = spark.sql("SELECT * FROM hrrr_bronze")
print("Table Schema:")
df.printSchema()

Table Schema:
root
 |-- date: date (nullable = true)
 |-- valid_time: timestamp (nullable = true)
 |-- latitude: float (nullable = true)
 |-- longitude: float (nullable = true)
 |-- sp: float (nullable = true)
 |-- gust: float (nullable = true)
 |-- vis: float (nullable = true)
 |-- prate: float (nullable = true)
 |-- crain: float (nullable = true)
 |-- csnow: float (nullable = true)
 |-- cape: float (nullable = true)
 |-- cin: float (nullable = true)
 |-- blh: float (nullable = true)
 |-- sdswrf: float (nullable = true)
 |-- sdlwrf: float (nullable = true)
 |-- suswrf: float (nullable = true)
 |-- sulwrf: float (nullable = true)
 |-- ishf: float (nullable = true)
 |-- slhtf: float (nullable = true)
 |-- gflux: float (nullable = true)
 |-- t_2m: float (nullable = true)
 |-- dpt: float (nullable = true)
 |-- u: float (nullable = true)
 |-- v: float (nullable = true)
 |-- source_path: string (nullable = true)
 |-- decode_error: string (nullable = true)



In [10]:
import pyspark.sql.functions as F
df_capitals = spark.read.format("delta").load(CAPITALS_TABLE_PATH)

bbox_df = df_capitals.filter(F.col("city") == "Phoenix")

phoenix = df.join(
    F.broadcast(bbox_df),
    (df["latitude"] >= bbox_df["min_lat"])
    & (df["latitude"] <= bbox_df["max_lat"])
    & (df["longitude"] >= bbox_df["min_lon"])
    & (df["longitude"] <= bbox_df["max_lon"]),
)

phoenix.show(truncate=False)


+----------+-------------------+---------+---------+-------+---------+-------+-----+-----+-----+------+------+---------+------+------+---------+---------+-----+-----+-----+---------+---------+---------+-----------+---------------------------------------------------------------------+------------+-------+-----+-------+-------+-------+-------+------------------+-------+
|date      |valid_time         |latitude |longitude|sp     |gust     |vis    |prate|crain|csnow|cape  |cin   |blh      |sdswrf|sdlwrf|suswrf   |sulwrf   |ishf |slhtf|gflux|t_2m     |dpt      |u        |v          |source_path                                                          |decode_error|city   |state|lat    |lon    |min_lat|max_lat|min_lon           |max_lon|
+----------+-------------------+---------+---------+-------+---------+-------+-----+-----+-----+------+------+---------+------+------+---------+---------+-----+-----+-----+---------+---------+---------+-----------+--------------------------------------------

In [11]:
phoenix.groupBy('date').agg({"date": 'count'}).orderBy('date', ascending=False).show()

+----------+-----------+
|      date|count(date)|
+----------+-----------+
|2026-09-28|      17020|
|2026-09-27|      17760|
|2026-09-26|      14800|
|2026-09-25|      17760|
|2026-09-24|      17760|
|2026-09-23|      17760|
|2026-09-22|      17760|
|2026-09-21|      17760|
|2026-09-20|      17760|
|2026-09-19|      17760|
|2026-09-18|      17760|
|2026-09-17|      17760|
|2026-09-16|      17760|
|2026-09-15|      17760|
|2026-09-14|      17760|
|2026-09-13|      17760|
|2026-09-12|      17760|
|2026-09-11|      17760|
|2026-09-10|      17760|
|2026-09-09|      17760|
+----------+-----------+
only showing top 20 rows



In [12]:
import pyspark.sql.functions as F

# 1. Load the tables directly from the Spark catalog
df_silver = spark.table("hrrr_silver")
df_capitals = spark.table("us_capitals")
df_silver_filtered = df_silver.filter(F.col("date") == "2026-09-24") 

joined = df_silver_filtered.join(
    F.broadcast(df_capitals),
    (df_silver_filtered["latitude"] >= df_capitals["min_lat"]) & 
    (df_silver_filtered["latitude"] <= df_capitals["max_lat"]) & 
    (df_silver_filtered["longitude"] >= df_capitals["min_lon"]) & 
    (df_silver_filtered["longitude"] <= df_capitals["max_lon"])
)

gold_daily = joined.groupBy("city", "state", "date").agg(
    F.round(F.avg("t_2m"), 2).alias("avg_temp_k"),
    F.round(F.max("t_2m"), 2).alias("max_temp_k"),
    F.round(F.min("t_2m"), 2).alias("min_temp_k"),
    F.round(F.avg("cape"), 2).alias("avg_cape"),
    F.round(F.max("gust"), 2).alias("max_gust"),
)

display(gold_daily.orderBy("city").toPandas())


,city,state,date,avg_temp_k,max_temp_k,min_temp_k,avg_cape,max_gust
0,Albany,NY,2026-09-24,284.32,294.149994,275.230011,0.01,12.540000
1,Annapolis,MD,2026-09-24,288.63,293.410004,282.359985,4.75,17.480000
2,Atlanta,GA,2026-09-24,292.20,297.279999,288.390015,16.38,11.850000
3,Augusta,ME,2026-09-24,284.88,296.239990,274.480011,0.00,8.110000
4,Austin,TX,2026-09-24,301.21,309.380005,293.140015,1207.47,10.890000
5,Baton Rouge,LA,2026-09-24,300.52,306.880005,294.510010,458.66,9.680000
6,Bismarck,ND,2026-09-24,290.69,299.559998,286.709991,38.06,18.799999
7,Boise,ID,2026-09-24,294.28,307.549988,282.609985,5.09,17.059999
8,Boston,MA,2026-09-24,285.93,292.660004,277.140015,3.73,13.390000
9,Carson City,NV,2026-09-24,290.73,304.750000,275.450012,14.84,12.920000


In [10]:
all_comments = {
    "date": "Forecast Date",
    "valid_time": "Forecast Valid Time",
    "latitude": "Latitude (Degrees)",
    "longitude": "Longitude (Degrees, 0 to 360)",
    "sp": "Surface Pressure [Pa]",
    "gust": "Surface Wind Gust [m/s]",
    "vis": "Visibility [m]",
    "prate": "Precipitation Rate [kg/m^2/s]",
    "crain": "Categorical Rain (yes=1; no=0)",
    "csnow": "Categorical Snow (yes=1; no=0)",
    "cape": "Convective Available Potential Energy [J/kg]",
    "cin": "Convective Inhibition [J/kg]",
    "blh": "Boundary Layer Height [m]",
    "sdswrf": "Downward Short-Wave Radiation Flux [W/m^2]",
    "sdlwrf": "Downward Long-Wave Radiation Flux [W/m^2]",
    "suswrf": "Upward Short-Wave Radiation Flux [W/m^2]",
    "sulwrf": "Upward Long-Wave Radiation Flux [W/m^2]",
    "ishf": "Ice Surface Heat Flux [W/m^2]",
    "slhtf": "Latent Heat Flux [W/m^2]",
    "gflux": "Ground Heat Flux [W/m^2]",
    "t_2m": "Temperature at 2m [K]",
    "dpt": "Dew Point Temperature at 2m [K]",
    "u": "U (X)-Component of Wind at 10m [m/s]",
    "v": "V (Y)-Component of Wind at 10m [m/s]",
    "source_path": "S3 path of the raw GRIB2 file",
    "decode_error": "GRIB2 decoding failure"
}

for col_name, comment in all_comments.items():
    spark.sql(f"ALTER TABLE hrrr_silver ALTER COLUMN {col_name} COMMENT '{comment}'")
